In [106]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import cumulative_trapezoid

from code_descriptors_postural_control.stabilogram.stato import Stabilogram
from code_descriptors_postural_control.descriptors import compute_all_features

In [107]:
# df = pd.read_csv("../data/antoine/radar24/test3.csv")
df = pd.read_csv("../results/kelly/X_vitesse.csv")
x_time = df["Time_s"].values
x_velocity = df["Vmax_m_s"].values
x_pos = cumulative_trapezoid(x_velocity, x_time, initial=0)


# df = pd.read_csv("../data/antoine/radar9/test3.csv")
df = pd.read_csv("../results/kelly/Y_vitesse.csv")
y_time = df["Time_s"].values
y_velocity = df["Vmax_m_s"].values
y_pos = cumulative_trapezoid(y_velocity, y_time, initial=0)


In [108]:
positive_time = np.where((x_time >= 0) & (x_time <= 10))
x_pos = x_pos[positive_time]
x_time = x_time[positive_time]

In [ ]:
common_times = np.linspace(0, 10, len(y_pos))
signal_x_resampled = np.interp(common_times, x_time, x_pos)
signal_y_resampled = np.interp(common_times, y_time, y_pos)

x_pos = signal_x_resampled
y_pos = signal_y_resampled
x_time = common_times
y_time = common_times

In [ ]:
common_times

In [ ]:
x_pos = x_pos*1000
y_pos = y_pos*1000

In [ ]:
1/x_time[1]

In [ ]:

plt.figure()
plt.suptitle("Radar Stabilogram")

# Plot X
plt.subplot(2, 1, 1)
plt.plot(x_time, x_pos)
plt.xlabel("Time (s)")
plt.ylabel("ML (mm)")
plt.title("ML")
plt.grid()

# Plot Y
plt.subplot(2, 1, 2)
plt.plot(y_time, y_pos)
plt.xlabel("Time (s)")
plt.ylabel("AP (mm)")
plt.title("AP")
plt.grid()

plt.tight_layout()
plt.show()


In [ ]:

plt.figure()
sc = plt.scatter(x_pos, y_pos, c=y_time, cmap='viridis', s=10)
plt.xlabel("ML (mm)")
plt.ylabel("AP (mm)")
plt.title("Radar Statokinesigram")
plt.grid()

cbar = plt.colorbar(sc)
cbar.set_label("Time (s)")
plt.show()

# Centering

In [ ]:
x_pos = x_pos - np.mean(x_pos)
y_pos = y_pos - np.mean(y_pos)

plt.figure()
plt.suptitle("Radar Stabilogram Centered")

# Plot X
plt.subplot(2, 1, 1)
plt.plot(y_time, x_pos)
plt.xlabel("Time (s)")
plt.ylabel("ML (mm)")
plt.title("Centered ML")
plt.grid()

# Plot Y
plt.subplot(2, 1, 2)
plt.plot(y_time, y_pos)
plt.xlabel("Time (s)")
plt.ylabel("AP (mm)")
plt.title("Centered ML")
plt.grid()

plt.tight_layout()
plt.show()

plt.figure()
sc = plt.scatter(x_pos, y_pos, c=y_time, cmap='viridis', s=10)
plt.xlabel("ML (mm)")
plt.ylabel("AP (mm)")
plt.title("Radar Statokinesigram Centered")
plt.grid()

cbar = plt.colorbar(sc)
cbar.set_label("Time (s)")
plt.show()

In [ ]:
x = x_pos - np.mean(x_pos)
y = y_pos - np.mean(y_pos)

modulus = np.sqrt(x**2 + y**2)
angle = np.arctan2(y, x)

modulus95 = modulus[modulus <= np.percentile(modulus, 95)]
rmax = np.max(modulus95)

mod_norm = modulus / rmax
x_norm = mod_norm * np.cos(angle)
y_norm = mod_norm * np.sin(angle)

In [ ]:
plt.figure()
sc = plt.scatter(x_norm, y_norm, c=y_time, cmap='viridis', s=10)
plt.xlabel("ML (mm)")
plt.ylabel("AP (mm)")
plt.title("Radar Statokinesigram Normalized testing")
plt.grid()

cbar = plt.colorbar(sc)
cbar.set_label("Time (s)")
plt.show()

In [ ]:
data = np.array([x_pos/10, y_pos/10]).T

stato = Stabilogram()
stato.from_array(array=data, original_frequency=1/y_time[1])

In [ ]:
sway_density_radius = 0.3 # 3 mm
params_dic = {"sway_density_radius": sway_density_radius}
features = compute_all_features(stato, params_dic=params_dic)

plt.plot(stato.medio_lateral)
plt.plot(stato.antero_posterior)

In [ ]:
features 

In [111]:
def get_descriptors(x_velocity_file, y_velocity_file):

    df = pd.read_csv(x_velocity_file)
    x_time = df["Time_s"].values
    x_velocity = df["Vmax_m_s"].values
    x_pos = cumulative_trapezoid(x_velocity, x_time, initial=0)


    df = pd.read_csv(y_velocity_file)
    y_time = df["Time_s"].values
    y_velocity = df["Vmax_m_s"].values
    y_pos = cumulative_trapezoid(y_velocity, y_time, initial=0)

    if (len(x_pos) != len(y_pos)):
        positive_time = np.where((x_time >= 0) & (x_time <= 10))
        x_pos = x_pos[positive_time]
        x_time = x_time[positive_time]

        common_times = np.linspace(0, 10, len(y_pos))
        signal_x_resampled = np.interp(common_times, x_time, x_pos)
        signal_y_resampled = np.interp(common_times, y_time, y_pos)

        x_pos = signal_x_resampled
        y_pos = signal_y_resampled
        x_time = common_times
        y_time = common_times

    x_pos = x_pos*1000
    y_pos = y_pos*1000

    x_pos = x_pos - np.mean(x_pos)
    y_pos = y_pos - np.mean(y_pos)

    data = np.array([x_pos/10, y_pos/10]).T

    stato = Stabilogram()
    stato.from_array(array=data, original_frequency=1/y_time[1])
    sway_density_radius = 0.3 # 3 mm
    params_dic = {"sway_density_radius": sway_density_radius}
    features = compute_all_features(stato, params_dic=params_dic)

    return features 


In [114]:
# file_list = [
#     ["../data/antoine/radar24/test1.csv", "../data/antoine/radar9/test1.csv"],
#     ["../data/antoine/radar24/test2.csv", "../data/antoine/radar9/test2.csv"],
#     ["../data/antoine/radar24/test3.csv", "../data/antoine/radar9/test3.csv"],
#     ["../data/antoine/radar24/test4.csv", "../data/antoine/radar9/test4.csv"],
#     ["../results/kelly/X_vitesse.csv", "../results/kelly/Y_vitesse.csv"]
# ]


file_list = [
    ["../results/antoine/test1_x_vitesse.csv", "../results/antoine/test1_y_vitesse.csv"],
    ["../results/antoine/test2_x_vitesse.csv", "../results/antoine/test2_y_vitesse.csv"],
    ["../results/antoine/test3_x_vitesse.csv", "../results/antoine/test3_y_vitesse.csv"],
    ["../results/antoine/test4_x_vitesse.csv", "../results/antoine/test4_y_vitesse.csv"],
    ["../results/kelly/X_vitesse.csv", "../results/kelly/Y_vitesse.csv"]
]

features_list = []
for x_file, y_file in file_list:
    features = get_descriptors(x_file, y_file)
    features_list.append(features)

df_features = pd.DataFrame(features_list)
df_features.to_csv("../results/radar_descriptors.csv")



C:\Users\Kelly\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\scipy\signal\_spectral_py.py:790: UserWarning: nperseg = 250 is greater than input length  = 246, using nperseg = 246
  freqs, _, Pxy = _spectral_helper(x, y, fs, window, nperseg, noverlap,
C:\Users\Kelly\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\scipy\signal\_spectral_py.py:790: UserWarning: nperseg = 250 is greater than input length  = 246, using nperseg = 246
  freqs, _, Pxy = _spectral_helper(x, y, fs, window, nperseg, noverlap,
C:\Users\Kelly\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\scipy\signal\_spectral_py.py:790: UserWarning: nperseg = 250 is greater than input length  = 246, using nperseg = 246
  freqs, _, Pxy = _spectral_helper(x, y, fs, window, nperseg, noverlap,
C:\Users\Kell

In [115]:
df_features

,mean_value_ML,mean_value_AP,mean_distance_ML,mean_distance_AP,mean_distance_Radius,maximal_distance_ML,maximal_distance_AP,maximal_distance_Radius,rms_ML,rms_AP,...,critical_time_Diffusion_ML,critical_displacement_Diffusion_ML,short_time_scaling_Diffusion_ML,long_time_scaling_Diffusion_ML,short_time_diffusion_Diffusion_AP,long_time_diffusion_Diffusion_AP,critical_time_Diffusion_AP,critical_displacement_Diffusion_AP,short_time_scaling_Diffusion_AP,long_time_scaling_Diffusion_AP
0,6.802523e-16,6.674173e-16,2.471614,2.676966,4.026234,5.884989,6.215051,6.959485,3.062702,3.159359,...,1.811484,50.050313,0.975451,-1.206527,17.241899,623.748417,1.976508,64.909311,0.972836,-1.660539
1,4.298103e-16,-1.013961e-16,2.061188,2.294392,3.455211,4.773069,5.617683,5.619100,2.382996,2.834845,...,1.907765,44.891554,0.942628,-1.269972,9.989441,82.800325,1.839378,32.576975,0.969826,-0.765330
2,8.984464e-18,3.953164e-16,2.284546,2.146374,3.449760,5.585507,4.995654,6.152549,2.719578,2.549349,...,1.738813,25.761400,0.970265,-0.252312,10.833868,77.030626,1.687552,29.939801,0.971290,-0.902974
3,3.593786e-17,4.004504e-16,1.395484,2.474225,3.083428,3.271625,4.842895,4.948512,1.661343,2.836342,...,1.931935,27.676742,0.971712,-1.929868,11.638028,177.476046,2.053700,37.757146,0.817695,-1.075298
4,3.450034e-15,-1.630038e-16,20.899148,1.407038,21.030320,53.183567,3.338202,53.199302,23.830867,1.719054,...,1.122450,936.790992,0.910986,0.358239,2.289426,1.857298,0.816962,1.586154,0.907655,0.390305
